# 09 · Handling Missing Data (NaN)

SQL has `NULL`. Pandas has `NaN`/`None`. This notebook covers `dropna`, `fillna`,
`isna`, `replace`, and casting types with `astype` — all on a small, easy-to-read
example DataFrame before you use them on a real dataset in the next notebook.

In [ ]:
import pandas as pd
import numpy as np

people = {
    'first': ['Corey', 'Jane', 'John', 'Chris', np.nan, None, 'NA'],
    'last': ['Schafer', 'Doe', 'Doe', 'Schafer', np.nan, np.nan, 'Missing'],
    'email': ['CoreyMSchafer@gmail.com', 'JaneDoe@email.com', 'JohnDoe@email.com',
              None, np.nan, 'Anonymous@email.com', 'NA'],
    'age': ['33', '55', '63', '36', None, None, 'Missing']
}
# SQL: CREATE TABLE people (first TEXT, last TEXT, email TEXT, age TEXT);
#      INSERT INTO people VALUES
#        ('Corey','Schafer','CoreyMSchafer@gmail.com','33'),
#        ('Jane','Doe','JaneDoe@email.com','55'),
#        ('John','Doe','JohnDoe@email.com','63'),
#        ('Chris','Schafer',NULL,'36'),
#        (NULL,NULL,NULL,NULL),
#        (NULL,NULL,'Anonymous@email.com',NULL),
#        ('NA','Missing','NA','Missing');


In [ ]:
df = pd.DataFrame(people)

df.replace('NA', np.nan, inplace=True)
df.replace('Missing', np.nan, inplace=True)
# The raw data uses the literal TEXT 'NA' and 'Missing' as placeholders for missing values —
# pandas doesn't know that on its own, so we convert those placeholder strings into real NaN.
# SQL: UPDATE people SET first = NULL WHERE first = 'NA';
#      UPDATE people SET last  = NULL WHERE last  = 'Missing';
#      -- ...repeated per column/placeholder value


In [ ]:
df
# SQL: SELECT * FROM people;


In [ ]:
df.dropna()   # drops any row that has AT LEAST ONE missing value, in ANY column
# SQL: SELECT * FROM people
#      WHERE first IS NOT NULL AND last IS NOT NULL AND email IS NOT NULL AND age IS NOT NULL;


In [ ]:
df.dropna(axis='index', how='all', subset=['last', 'email'])
# axis='index'  -> drop ROWS (not columns)
# how='all'     -> only drop a row if EVERY column in `subset` is missing
# subset=[...]  -> only look at these columns when deciding
# SQL: SELECT * FROM people WHERE NOT (last IS NULL AND email IS NULL);


In [ ]:
df.isna()   # True/False mask showing exactly where the missing values are
# SQL: SELECT first IS NULL, last IS NULL, email IS NULL, age IS NULL FROM people;


In [ ]:
df.fillna(0)   # replace every remaining NaN with 0 (just for this preview, not saved back)
# SQL: SELECT COALESCE(first, 0), COALESCE(last, 0), COALESCE(email, 0), COALESCE(age, 0) FROM people;
# -- to persist it: UPDATE people SET first = COALESCE(first, 0), last = COALESCE(last, 0), ...;


In [ ]:
df.dtypes
# SQL: SELECT column_name, data_type FROM information_schema.columns WHERE table_name = 'people';


---
**🐛 Bug fix — `df['age'].mean()` throws `TypeError`**

```python
df['age'].mean()
```
throws:
```
TypeError: can only concatenate str (not "int") to str
```
Even though the values *look* like numbers ('33', '55', ...), pandas read this column
as **text** (`dtype: object`), since it came from a Python list of strings. You can't
average text. Fix: convert the column to a numeric type with `.astype(float)` first.

In [ ]:
df['age'] = df['age'].astype(float)
# SQL: ALTER TABLE people MODIFY COLUMN age FLOAT;
#      -- (MySQL: ALTER TABLE ... MODIFY COLUMN changes a column's data type in place)


In [ ]:
df.dtypes
# SQL: SELECT column_name, data_type FROM information_schema.columns WHERE table_name = 'people';


In [ ]:
df['age'].mean()   # now works, since age is numeric
# SQL: SELECT AVG(age) FROM people;
